# Data Initialization


In [1]:
import pandas as pd 
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

raw_df = pd.read_csv('weekly_player_stats_offense.csv')
rng = np.random.RandomState(seed=67)

In [2]:
# Holden: initial values for labels of features to be used or excluded by model
categorical_features = set([])
real_val_features = set([])
features_dropped = set([]) # remove from lists of kept features
target_labels = set(['fantasy_points_ppr'])

# Holden
for col in raw_df.columns :
    if pd.api.types.is_numeric_dtype(raw_df[col]) :
        real_val_features.add(col)
    else :
        categorical_features.add(col)

features_dropped.update(['player_id', 'player_name']) # categorical features not relevant to prediction

target_labels = set(['fantasy_points_ppr'])

# Holden
for col in raw_df.columns :
    if pd.api.types.is_numeric_dtype(raw_df[col]) :
        real_val_features.add(col)
    else :
        categorical_features.add(col)

features_dropped.update(['player_id', 'player_name']) # categorical features not relevant to prediction

# Holden: add all fantasy point statistics to list of target labels
for label in raw_df :
    label = str(label)
    if 'fantasy' in label :
        target_labels.add(label)


In [3]:
# Holden and Connor
# add features with more than missing_threshold% missing data to drop features_list
temp = raw_df.drop(features_dropped, axis=1)
missing_values = temp.columns[temp.isnull().any()]
missing_fractions = raw_df[list(missing_values)].isnull().mean() 

missing_threshold = missing_fractions.mean()
features_dropped.update(missing_fractions[missing_fractions > missing_threshold].index)

In [4]:
from sklearn.preprocessing import OneHotEncoder,  StandardScaler, RobustScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer

# ensure feature lists are cohesive
for label in features_dropped :
    if label in categorical_features :
        categorical_features.remove(label)
    elif label in real_val_features :
        real_val_features.remove(label)

real_val_features.difference_update(target_labels) # take potential targets out of features processed by pipeline

# convert to lists so we can stop typcasting 
real_val_features = list(real_val_features) 
categorical_features = list(categorical_features)
target_labels = list(target_labels)
features_dropped = list(features_dropped)

# clean dataframe without features exceeding missing value threshold
clean_df = raw_df.drop(list(features_dropped), axis=1) 


In [5]:

# sort real value features by estimated distribution type
rv_split_std = []
rv_split_skewed = []
rv_split_few = []
for feature in real_val_features :
    column = clean_df[feature]
    unique_vals = column.nunique(dropna=True)

    if unique_vals <= 2 :
        rv_split_few.append(feature)
    else :
        skew = column.skew(skipna=True)
        if abs(skew) > 1 :
            rv_split_skewed.append(feature)
        else :
            rv_split_std.append(feature)

# data preprocessing pipelines
# each distribution gets the appropriate scaler 
# current system expects most values to be skewed 
#   - we may be able to improve our estimation and normalizatioin
std_pipline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

skewed_pipline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', RobustScaler())
])

no_scale_pipeline = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')) 
])

categorical_pipeline = Pipeline(steps=[
    ('OHE', OneHotEncoder(handle_unknown='ignore'))
])

preprocess = ColumnTransformer(transformers=[
    ('norm_rv', std_pipline, rv_split_std),
    ('skewed_rv', skewed_pipline, rv_split_skewed),
    ('few_rv', no_scale_pipeline, rv_split_few),
    ('categorical', categorical_pipeline, categorical_features)
])

In [ ]:
# Connor: Code implementation of a Random Forest model that can predict fantasy points based on player stats
# Holden: heavily refined model implementation
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
	
def train_and_evaluate_model(position_df, position_name) :
	
	X = position_df.drop(columns=list(target_labels))
	y = position_df['fantasy_points_ppr']
	
	X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=rng)
	
	X_train = preprocess.fit_transform(X_train)
	X_test = preprocess.transform(X_test)

	model = RandomForestRegressor(n_estimators=100, random_state=rng, n_jobs=-1)
	model.fit(X_train, y_train)
	
	y_pred = model.predict(X_test)
	
	mse = mean_squared_error(y_test, y_pred)
	r2 = r2_score(y_test, y_pred)
	
	print(f"Position: {position_name}")
	print(f"Mean Squared Error: {mse}")
	print(f"R^2 Score: {r2}\n")


In [7]:

# create seperate datasets for each position
relevant_positions = ['QB', 'RB', 'WR', 'TE']
filtered = clean_df[clean_df['position'].isin(relevant_positions)]
pos_filtered_dfs = {pos : pos_df for pos, pos_df in filtered.groupby('position')}

for position in relevant_positions :
    train_and_evaluate_model(pos_filtered_dfs[position], position)

Position: QB
Mean Squared Error: 1.0004512743415899
R^2 Score: 0.9918276342871258

Position: RB
Mean Squared Error: 0.4383110913524321
R^2 Score: 0.9931975056535909

Position: WR
Mean Squared Error: 0.248241652042452
R^2 Score: 0.9962105751714667

Position: TE
Mean Squared Error: 0.3227043783396544
R^2 Score: 0.9917939155565542



In [8]:
train_and_evaluate_model(clean_df, 'ALL')

Position: ALL
Mean Squared Error: 0.35354541511854004
R^2 Score: 0.9952886581971149

